In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import gdown
import numpy as np

In [3]:
!gdown --id 16CUDwVljbRk33oeG-w2YhuMcfQl7m9BZ -O X_train.npy

!gdown --id 1e3IYJuYUEHrxOPayxP8PlZ45YcK3vip4 -O X_test.npy

!gdown --id 1mF6szu7juZ5n75xOix5CfwwpbpklJIDj -O X_val.npy

!gdown --id 1HIywHpgBFW5wDdkr6bgaw4yzfZd3oS36 -O y_train.npy

!gdown --id 1qL6KE1XlJmF6epaKzoETb-nOiesZsr39 -O y_test.npy

!gdown --id 1sU9NmM_-qLERVlf02_03Q0yXi3QVg6AP -O y_val.npy

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=16CUDwVljbRk33oeG-w2YhuMcfQl7m9BZ
From (redirected): https://drive.google.com/uc?id=16CUDwVljbRk33oeG-w2YhuMcfQl7m9BZ&confirm=t&uuid=66680e5d-c37e-4175-ba62-7a1d7c444a98
To: /content/X_train.npy
100% 1.52G/1.52G [00:19<00:00, 75.9MB/s]
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1e3IYJuYUEHrxOPayxP8PlZ45YcK3vip4
From (redirected): https://drive.google.com/uc?id=1e3IYJuYUEHrxOPayxP8PlZ45YcK3vip4&confirm=t&uuid=c6cf44cd-2a48-4456-977e-d41114e21dce
To: /content/X_test.npy
10

In [4]:
X_train = np.load('X_train.npy')
y_train = np.load('y_train.npy')

print('X_train shape:', X_train.shape)
print('y_train shape:', y_train.shape)

X_train shape: (40000, 256, 37)
y_train shape: (40000,)


In [5]:
X_test = np.load('X_test.npy')
y_test = np.load('y_test.npy')

print('X_test shape:', X_test.shape)
print('y_test shape:', y_test.shape)

X_test shape: (5000, 256, 37)
y_test shape: (5000,)


In [6]:
X_val = np.load('X_val.npy')
y_val = np.load('y_val.npy')

print('X_val shape:', X_val.shape)
print('y_val shape:', y_val.shape)

X_val shape: (5000, 256, 37)
y_val shape: (5000,)


In [7]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(enumerate(class_weights))
print(class_weights)

{0: np.float64(0.5292685508627077), 1: np.float64(9.041591320072333)}


# Task
Build, compile, and train a Bidirectional LSTM model for binary classification. The model should accept inputs of shape (256, 37), use the Adam optimizer, binary crossentropy loss, and 'accuracy' as a metric. Implement EarlyStopping to monitor 'val_accuracy' with a suitable patience, and train the model using the provided `X_train`, `y_train`, `X_val`, `y_val` datasets, and `class_weights`. Finally, summarize the training process, including the final validation accuracy and any observations about convergence.

## Build BiLSTM Model

### Subtask:
Define the architecture for a Bidirectional LSTM model, considering the input shape (256, 37) and the output being binary classification. The model should include LSTM layers and appropriate dense layers for classification.


**Reasoning**:
To define the Bidirectional LSTM model architecture, I need to import the required layers from TensorFlow Keras, specify the input shape, add a Bidirectional LSTM layer, and a Dense output layer for binary classification.



In [8]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Bidirectional, LSTM, Dense

# Define the input layer with shape (timesteps, features)
input_layer = Input(shape=(256, 37))

# Add a Bidirectional LSTM layer
# Using 128 units for the LSTM layer and setting return_sequences=False as it's followed by a Dense layer
bilstm_layer = Bidirectional(LSTM(128, return_sequences=False))(input_layer)

# Add a Dense output layer for binary classification
output_layer = Dense(1, activation='sigmoid')(bilstm_layer)

# Create the model
model = Model(inputs=input_layer, outputs=output_layer)

print("Bidirectional LSTM model architecture defined successfully.")

Bidirectional LSTM model architecture defined successfully.


## Compile Model

### Subtask:
Compile the BiLSTM model with an appropriate optimizer (e.g., Adam), a suitable loss function for binary classification (e.g., binary_crossentropy), and 'accuracy' as a metric.


**Reasoning**:
To compile the model, I will use the `compile` method on the `model` object, specifying the Adam optimizer, binary crossentropy loss, and accuracy as the metric. Importing `Adam` is good practice, though `'adam'` as a string is also acceptable.



In [9]:
from tensorflow.keras.optimizers import Adam

# Compile the model
model.compile(
    optimizer=Adam(learning_rate=0.001), # Using Adam optimizer with a default learning rate
    loss='binary_crossentropy',       # Binary crossentropy for binary classification
    metrics=['accuracy']              # Monitor accuracy during training
)

print("BiLSTM model compiled successfully.")

BiLSTM model compiled successfully.


**Reasoning**:
Now that the model architecture is defined and compiled, the next step is to train the model. This involves fitting the model to the training data, validating it with the validation data, and incorporating EarlyStopping for regularization and class weights to handle potential class imbalance. I will also capture the training history to analyze convergence.



In [10]:
from tensorflow.keras.callbacks import EarlyStopping

# Define EarlyStopping callback
early_stopping = EarlyStopping(
    monitor='val_accuracy', # Monitor validation accuracy
    patience=5,             # Number of epochs with no improvement after which training will be stopped
    restore_best_weights=True # Restore model weights from the epoch with the best value of the monitored quantity
)

# Train the model
history = model.fit(
    X_train, y_train,
    epochs=20, # Start with a reasonable number of epochs, EarlyStopping will prevent overfitting
    batch_size=32, # Common batch size
    validation_data=(X_val, y_val),
    class_weight=class_weights, # Apply class weights to handle imbalance
    callbacks=[early_stopping], # Include the EarlyStopping callback
    verbose=1 # Show training progress
)

print("BiLSTM model training initiated.")

Epoch 1/20
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 37s 27ms/step - accuracy: 0.5241 - loss: 0.7006 - val_accuracy: 0.4910 - val_loss: 0.6987
Epoch 2/20
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 31s 25ms/step - accuracy: 0.5422 - loss: 0.6799 - val_accuracy: 0.6140 - val_loss: 0.6581
Epoch 3/20
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 40s 24ms/step - accuracy: 0.6158 - loss: 0.6466 - val_accuracy: 0.6918 - val_loss: 0.5909
Epoch 4/20
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 29s 24ms/step - accuracy: 0.7547 - loss: 0.5115 - val_accuracy: 0.6614 - val_loss: 0.6139
Epoch 5/20
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 30s 24ms/step - accuracy: 0.8605 - loss: 0.3107 - val_accuracy: 0.8092 - val_loss: 0.4512
Epoch 6/20
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 41s 24ms/step - accuracy: 0.9422 - loss: 0.1435 - val_accuracy: 0.8632 - val_loss: 0.4078
Epoch 7/20
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 29s 24ms/step - accuracy: 0.9793 - loss: 0.0625 - val_accuracy: 0.8934 - val_loss: 0.4227
Epoch 8/20
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 29s 23ms/step - accuracy: 0.9896 -

In [11]:
from sklearn.metrics import accuracy_score
y_pred = model.predict(X_test)
y_pred = (y_pred > 0.5).astype(int)
acc = accuracy_score(y_test, y_pred)
print("Test Accuracy:", acc)

157/157 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step
Test Accuracy: 0.913


In [13]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional

model2 = Sequential()

model2.add(Bidirectional(LSTM(64, return_sequences=True), input_shape=(256, 37)))
model2.add(Dropout(0.3))

model2.add(Bidirectional(LSTM(32)))
model2.add(Dropout(0.3))

model2.add(Dense(16, activation='relu'))
model2.add(Dense(1, activation='sigmoid'))

model2.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [15]:
early_stopping = EarlyStopping(
    monitor='val_accuracy', # Monitor validation loss
    patience=5,             # Number of epochs with no improvement after which training will be stopped
    restore_best_weights=True # Restore model weights from the epoch with the best value of the monitored quantity
)

# Train the model
history2 = model2.fit(
    X_train, y_train,
    epochs=25, # Start with a reasonable number of epochs, EarlyStopping will prevent overfitting
    batch_size=64, # Common batch size
    validation_data=(X_val, y_val),
    class_weight=class_weights, # Apply class weights to handle imbalance
    callbacks=[early_stopping], # Include the EarlyStopping callback
    verbose=1 # Show training progress
)

Epoch 1/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 31s 43ms/step - accuracy: 0.5059 - loss: 0.6884 - val_accuracy: 0.5948 - val_loss: 0.6855
Epoch 2/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 26s 42ms/step - accuracy: 0.4875 - loss: 0.6859 - val_accuracy: 0.3658 - val_loss: 0.7072
Epoch 3/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 26s 42ms/step - accuracy: 0.5055 - loss: 0.6822 - val_accuracy: 0.3426 - val_loss: 0.7414
Epoch 4/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 26s 41ms/step - accuracy: 0.5514 - loss: 0.6731 - val_accuracy: 0.5854 - val_loss: 0.6677
Epoch 5/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 42s 43ms/step - accuracy: 0.5748 - loss: 0.6613 - val_accuracy: 0.6050 - val_loss: 0.6522
Epoch 6/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 40s 42ms/step - accuracy: 0.6418 - loss: 0.6141 - val_accuracy: 0.6478 - val_loss: 0.6071
Epoch 7/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 25s 41ms/step - accuracy: 0.7185 - loss: 0.5222 - val_accuracy: 0.6240 - val_loss: 0.6382
Epoch 8/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 25s 41ms/step - accuracy: 0.7619 - loss: 0.4472 - 

In [17]:
from sklearn.metrics import accuracy_score
y_pred = model2.predict(X_test)
y_pred = (y_pred > 0.5).astype(int)
acc = accuracy_score(y_test, y_pred)
print("Test Accuracy:", acc)

157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step
Test Accuracy: 0.7552


In [20]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional

model3 = Sequential()

model3.add(Bidirectional(LSTM(64, return_sequences=True), input_shape=(256, 37)))
model3.add(Dropout(0.3))

model3.add(Bidirectional(LSTM(32)))
model3.add(Dropout(0.3))

model3.add(Dense(16, activation='relu'))
model3.add(Dense(1, activation='sigmoid'))

model3.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
early_stopping = EarlyStopping(
    monitor='val_accuracy', # Monitor validation accuracy
    patience=5,             # Number of epochs with no improvement after which training will be stopped
    restore_best_weights=True # Restore model weights from the epoch with the best value of the monitored quantity
)

# Train the model
history3 = model3.fit(
    X_train, y_train,
    epochs=25, # Start with a reasonable number of epochs, EarlyStopping will prevent overfitting
    batch_size=64, # Common batch size
    validation_data=(X_val, y_val),
    class_weight=class_weights, # Apply class weights to handle imbalance
    callbacks=[early_stopping], # Include the EarlyStopping callback
    verbose=1 # Show training progress
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/25
625/625 ━━━━━━━━━━━━━━━━━━━━ 34s 47ms/step - accuracy: 0.5107 - loss: 0.7013 - val_accuracy: 0.3218 - val_loss: 0.7065
Epoch 2/25
625/625 ━━━━━━━━━━━━━━━━━━━━ 27s 43ms/step - accuracy: 0.4565 - loss: 0.6980 - val_accuracy: 0.6106 - val_loss: 0.6709
Epoch 3/25
625/625 ━━━━━━━━━━━━━━━━━━━━ 26s 41ms/step - accuracy: 0.5450 - loss: 0.6913 - val_accuracy: 0.5242 - val_loss: 0.6980
Epoch 4/25
625/625 ━━━━━━━━━━━━━━━━━━━━ 26s 41ms/step - accuracy: 0.5532 - loss: 0.6770 - val_accuracy: 0.5932 - val_loss: 0.6731
Epoch 5/25
625/625 ━━━━━━━━━━━━━━━━━━━━ 26s 41ms/step - accuracy: 0.6294 - loss: 0.6472 - val_accuracy: 0.5986 - val_loss: 0.6696
Epoch 6/25
625/625 ━━━━━━━━━━━━━━━━━━━━ 26s 42ms/step - accuracy: 0.6398 - loss: 0.6182 - val_accuracy: 0.6644 - val_loss: 0.6172
Epoch 7/25
625/625 ━━━━━━━━━━━━━━━━━━━━ 26s 41ms/step - accuracy: 0.7152 - loss: 0.5416 - val_accuracy: 0.6806 - val_loss: 0.6055
Epoch 8/25
625/625 ━━━━━━━━━━━━━━━━━━━━ 26s 41ms/step - accuracy: 0.7648 - loss: 0.4624 - 

In [22]:
from sklearn.metrics import accuracy_score
y_pred = model3.predict(X_test)
y_pred = (y_pred > 0.5).astype(int)
acc = accuracy_score(y_test, y_pred)
print("Test Accuracy:", acc)

157/157 ━━━━━━━━━━━━━━━━━━━━ 7s 37ms/step
Test Accuracy: 0.868
